# Dữ liệu Marketing Campaign (Customer Personality Analysis)
**Nguồn trong đề:** Lab 1 — mục 1.1.2 nhiệm vụ 2 và mục 1.3 (ydata_profiling, dtale, SweetViz, AutoViz)

Notebook độc lập cho một tập dữ liệu; code lấy từ các ô tương ứng của notebook Lab1 (giữ nguyên, ô nào chỉnh có ghi `# [Chỉnh]`).

In [1]:
# Thư viện dùng chung (đề import rải rác ở nhiều ô; gom lại để notebook chạy độc lập)
import os
if os.path.basename(os.getcwd()) == 'notebooks':   # chạy notebook từ thư mục notebooks/ thì lùi ra thư mục gốc chứa data/
    os.chdir('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

#### Nhiệm vụ 2: Khám phá và xử lý dữ liệu Marketing Campaign

In [2]:
import pandas as pd
# [Chỉnh] file Marketing Campaign (Kaggle) phân cách bằng dấu Tab nên cần sep='\t'
marketing_data = pd.read_csv("data/marketing_campaign.csv", sep='\t', encoding='utf-8-sig')
# [Chỉnh] tên cột đúng là 'Dt_Customer' (đề in nhầm 'Dt_ Customer')
marketing_data = marketing_data[['ID','Year_Birth', 'Education',
 'Marital_Status','Income','Kidhome', 'Teenhome',
 'Dt_Customer', 'Recency','NumStorePurchases',
 'NumWebVisitsMonth']]

**2. Loại bỏ dữ liệu trùng lặp**

In [3]:
marketing_data.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,NumStorePurchases,NumWebVisitsMonth
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,4,7
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,2,5
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,10,4
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,4,6
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,6,5


In [4]:
# Remove duplicates across the columns in our dataset:
marketing_data_duplicate = marketing_data.drop_duplicates()
print(marketing_data.shape, '->', marketing_data_duplicate.shape)

(2240, 11) -> (2240, 11)


In [5]:
# Delete a specified row at index value 1:
marketing_data.drop(labels=[1], axis=0)

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,NumStorePurchases,NumWebVisitsMonth
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,4,7
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,10,4
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,4,6
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,6,5
5,7446,1967,Master,Together,62513.0,0,1,09-09-2013,16,10,6
...,...,...,...,...,...,...,...,...,...,...,...
2235,10870,1967,Graduation,Married,61223.0,0,1,13-06-2013,46,4,5
2236,4001,1946,PhD,Together,64014.0,2,1,10-06-2014,56,5,7
2237,7270,1981,Graduation,Divorced,56981.0,0,0,25-01-2014,91,13,6
2238,8235,1956,Master,Together,69245.0,0,1,24-01-2014,8,10,3


In [6]:
# Delete a single column
marketing_data.drop(labels=['Year_Birth'], axis=1)

,ID,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,NumStorePurchases,NumWebVisitsMonth
0,5524,Graduation,Single,58138.0,0,0,04-09-2012,58,4,7
1,2174,Graduation,Single,46344.0,1,1,08-03-2014,38,2,5
2,4141,Graduation,Together,71613.0,0,0,21-08-2013,26,10,4
3,6182,Graduation,Together,26646.0,1,0,10-02-2014,26,4,6
4,5324,PhD,Married,58293.0,1,0,19-01-2014,94,6,5
...,...,...,...,...,...,...,...,...,...,...
2235,10870,Graduation,Married,61223.0,0,1,13-06-2013,46,4,5
2236,4001,PhD,Together,64014.0,2,1,10-06-2014,56,5,7
2237,7270,Graduation,Divorced,56981.0,0,0,25-01-2014,91,13,6
2238,8235,Master,Together,69245.0,0,1,24-01-2014,8,10,3


**3. Thay thế dữ liệu và thay đổi định dạng của dữ liệu**

In [7]:
# Replace the values in Teenhome with has teen and has no teen
marketing_data['Teenhome_replaced'] = marketing_data['Teenhome'].replace([0,1,2],['has no teen','has teen','has teen'])
marketing_data[['Teenhome','Teenhome_replaced']].head()

,Teenhome,Teenhome_replaced
0,0,has no teen
1,1,has teen
2,0,has no teen
3,0,has no teen
4,0,has no teen


In [8]:
# Fill NAs in the Income column
marketing_data['Income'] = marketing_data['Income'].fillna(0)
# Change the data type of the Income column from float to int
marketing_data['Income_changed'] = marketing_data['Income'].astype(int)
marketing_data[['Income','Income_changed']].dtypes

Income            float64
Income_changed      int64
dtype: object

**4. Xử lý dữ liệu thiếu**

In [9]:
# Check for missing values using the isnull and sum methods
marketing_data.isnull().sum()

ID                   0
Year_Birth           0
Education            0
Marital_Status       0
Income               0
Kidhome              0
Teenhome             0
Dt_Customer          0
Recency              0
NumStorePurchases    0
NumWebVisitsMonth    0
Teenhome_replaced    0
Income_changed       0
dtype: int64

In [10]:
# Drop missing values using the dropna method
marketing_data_withoutna = marketing_data.dropna(how = 'any')
marketing_data_withoutna.shape

(2240, 13)

### Bài toán 3: công cụ EDA tự động
#### Nhiệm vụ 1: ydata_profiling (trước đây là pandas_profiling) trên dữ liệu Customer Personality Analysis

In [11]:
import pandas as pd
from ydata_profiling import ProfileReport
# [Chỉnh] sep='\t' cho đúng định dạng file Marketing Campaign
marketing_data = pd.read_csv("data/marketing_campaign.csv", sep='\t', encoding='utf-8-sig')
# Create an Automated EDA report using the ProfileReport class in the
# ydata_profiling library. Use the to_file method to output the report to an HTML file
profile = ProfileReport(marketing_data)
profile.to_file("Reports/profile_output.html")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/ipykernel_113/28472545.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/34 [00:00<?, ?it/s, Describe variable: ID]

  0%|          | 0/29 [00:00<?, ?it/s]

...

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML: 100%|██████████| 1/1 [00:04<00:00,  4.42s/it]

Render HTML: 100%|██████████| 1/1 [00:04<00:00,  4.43s/it]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 15.74it/s]

#### Nhiệm vụ 2: dtale trên dữ liệu Marketing Campaign
> dtale mở một web-server và trình duyệt, nên ô này không chạy tự động trong notebook báo cáo (chạy trên máy: `pip install dtale`).

In [ ]:
import pandas as pd
import dtale
marketing_data = pd.read_csv("data/marketing_campaign.csv", sep='\t', encoding='utf-8-sig')
dtale.show(marketing_data).open_browser()

### 1.3.3. Bài tập thực hành 1 — SweetViz trên tập Marketing Campaign
**Tính năng chính của SweetViz:**
- Tạo báo cáo HTML tương tác chỉ với một dòng lệnh `sv.analyze(df)`; mỗi biến có thống kê (số lượng, thiếu, distinct, mean/median/std…) cùng biểu đồ phân bố.
- Hiển thị ma trận tương quan/liên hệ giữa các biến, kể cả biến phân loại (hệ số tương quan, uncertainty coefficient, correlation ratio).
- So sánh hai tập dữ liệu bằng `sv.compare([train, 'Train'], [test, 'Test'])` hoặc so sánh hai nhóm con bằng `sv.compare_intra(df, df['Education']=='PhD', ['PhD','Others'])`.
- Phân tích theo biến mục tiêu (`target_feat='Response'`) để xem mối quan hệ của từng biến với nhãn.
- Xuất báo cáo bằng `show_html(...)` (file HTML độc lập) hoặc `show_notebook()`.

**Cách sử dụng:** `pip install sweetviz` → `report = sv.analyze(df)` → `report.show_html('file.html')`.

In [12]:
import sweetviz as sv
marketing_data = pd.read_csv("data/marketing_campaign.csv", sep='\t', encoding='utf-8-sig')
report = sv.analyze(marketing_data)
report.show_html("Reports/sweetviz_report.html", open_browser=False)

                                             |          | [  0%]   00:00 -> (? left)

[Summarizing dataframe]                      |          | [  0%]   00:00 -> (? left)

Feature: ID                                  |▎         | [  3%]   00:00 -> (00:00 left)

Feature: ID                                  |▋         | [  7%]   00:03 -> (00:47 left)

Feature: Year_Birth                          |▋         | [  7%]   00:03 -> (00:47 left)

...

[Step 2/3] Processing Pairwise Features      |          | [  0%]   00:00 -> (? left)    

[Step 3/3] Generating associations graph     |          | [  0%]   00:00 -> (? left)

Done! Use 'show' commands to display/save.   |          | [  0%]   00:08 -> (? left)

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:08 -> (00:00 left)

Report Reports/sweetviz_report.html was generated.


### 1.3.4. Bài tập thực hành 2 — AutoViz trên tập Marketing Campaign
**Tính năng chính của AutoViz:**
- Tự động vẽ nhiều loại biểu đồ (histogram, boxplot, scatter, pairplot, heatmap tương quan, violin, biểu đồ cột cho biến phân loại…) chỉ với một lệnh `AV.AutoViz(file)`; công cụ tự chọn biểu đồ phù hợp theo kiểu dữ liệu.
- Tự phát hiện và xử lý một số vấn đề dữ liệu như cột có tỉ lệ thiếu cao, cột hằng số, cột trùng, ngoại lai; có thể chỉ định biến mục tiêu `depVar`.
- Với tập lớn có thể lấy mẫu (`max_rows_analyzed`, `max_cols_analyzed`) để chạy nhanh; có thể xuất biểu đồ ra thư mục (`chart_format='png'`/`'html'`).

**Cách sử dụng:** `pip install autoviz` → `from autoviz.AutoViz_Class import AutoViz_Class` → `AV = AutoViz_Class(); AV.AutoViz('file.csv', sep='\t')`.
> AutoViz mở nhiều biểu đồ trong trình duyệt/notebook và thường xung đột phiên bản thư viện, nên ô code dưới đây chạy trên máy cá nhân.

In [ ]:
from autoviz.AutoViz_Class import AutoViz_Class
AV = AutoViz_Class()
dft = AV.AutoViz("data/marketing_campaign.csv", sep='\t')